# AgeLens — 06 EG-004 Creatinine Training-Scale Sensitivity

This notebook quantifies the unresolved **EG-004** question:

> D-001 was trained on NHANES III creatinine measurements that were reported approximately 0.11–0.23 mg/dL higher than a standardized enzymatic reference. What is the effect of applying the NHANES-III-trained formula to modern creatinine measurements without a compensating shift?

## Scope

The analysis evaluates four explicit scenarios while holding every other participant-level value constant:

| Scenario | Added to modern harmonized creatinine |
| --- | ---: |
| `observed_modern` | 0.00 mg/dL |
| `nhanes3_bias_low` | +0.11 mg/dL |
| `nhanes3_bias_mid` | +0.17 mg/dL |
| `nhanes3_bias_high` | +0.23 mg/dL |

The positive shifts approximate placing modern creatinine values onto the historically higher NHANES III measurement scale. They are **sensitivity scenarios**, not adopted corrections.

## Governance safeguard

- EG-004 remains open.
- No scenario is made primary.
- No authoritative governance document is edited.
- Mortality data are not used.
- All outputs remain diagnostic and cannot be reported as final scientific results.


In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_FOLDER_NAME = "nhanes"

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 220)

print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"Current working directory: {Path.cwd().resolve()}")


numpy: 2.4.6
pandas: 2.3.3
Current working directory: <PROJECT_ROOT>\notebooks


In [2]:
def find_project_root(
    folder_name: str = PROJECT_FOLDER_NAME,
) -> Path:
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if candidate.name.lower() == folder_name.lower():
            return candidate

    raise FileNotFoundError(
        f"Could not find a parent folder named '{folder_name}'."
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = (
    PROJECT_ROOT / "configs" / "agelens_config.json"
)

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Configuration not found: {CONFIG_PATH}"
    )

CONFIG = json.loads(
    CONFIG_PATH.read_text(encoding="utf-8")
)

INTERIM_ROOT = (
    PROJECT_ROOT / CONFIG["paths"]["interim_data"]
)
PROCESSED_ROOT = (
    PROJECT_ROOT / CONFIG["paths"]["processed_data"]
)
TABLES_ROOT = PROJECT_ROOT / CONFIG["paths"]["tables"]
LOGS_ROOT = PROJECT_ROOT / CONFIG["paths"]["logs"]
DOCS_METHODOLOGY_ROOT = (
    PROJECT_ROOT / "docs" / "methodology"
)

for path in [
    PROCESSED_ROOT,
    TABLES_ROOT,
    LOGS_ROOT,
    DOCS_METHODOLOGY_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)

INPUT_PATH = (
    INTERIM_ROOT
    / "nhanes_2015_2018_preprocessed_diagnostic.parquet"
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input: {INPUT_PATH}")


Project root: <PROJECT_ROOT>
Input: <PROJECT_ROOT>\data\interim\nhanes_2015_2018_preprocessed_diagnostic.parquet


## 1. Load the harmonized complete-case sample

The sensitivity is performed on the same positive-weight, harmonized complete-case participants used by notebooks 03–05.


In [3]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Preprocessed input not found: {INPUT_PATH}"
    )

data = pd.read_parquet(INPUT_PATH)

required_columns = {
    "SEQN",
    "NHANES_CYCLE",
    "chronological_age_years",
    "age_topcoded",
    "age_below_20",
    "WTSAF4YR",
    "SDMVSTRA",
    "SDMVPSU",
    "complete_case_harmonized",
    "albumin_harmonized_g_L",
    "creatinine_harmonized_mg_dL",
    "creatinine_harmonized_umol_L",
    "glucose_mmol_L",
    "log_crp_harmonized",
    "lymphocyte_percent",
    "mcv_fL",
    "rdw_percent",
    "alp_harmonized_U_L",
    "wbc_1000cells_uL",
    "harmonized_phenoage_erratum_years",
    "harmonized_phenoage_supplement_years",
}

missing_columns = sorted(
    required_columns - set(data.columns)
)

if missing_columns:
    raise ValueError(
        f"Input is missing required columns: {missing_columns}"
    )

open_gaps = set(
    CONFIG["governance"]["open_core_evidence_gaps"]
)

if "EG-004" not in open_gaps:
    raise RuntimeError(
        "EG-004 is not recorded as an open Core Evidence Gap. "
        "This diagnostic notebook must not run after silent "
        "governance changes."
    )

analysis = data.loc[
    data["complete_case_harmonized"]
    & data["WTSAF4YR"].notna()
    & data["WTSAF4YR"].gt(0)
].copy()

if analysis.empty:
    raise RuntimeError(
        "The harmonized positive-weight complete-case sample is empty."
    )

if analysis.duplicated(
    ["NHANES_CYCLE", "SEQN"]
).any():
    raise RuntimeError(
        "Duplicate cycle + SEQN rows in the analysis sample."
    )

analysis["pooled_stratum"] = (
    analysis["NHANES_CYCLE"].astype(str)
    + "__"
    + analysis["SDMVSTRA"].astype(str)
)
analysis["pooled_psu"] = (
    analysis["NHANES_CYCLE"].astype(str)
    + "__"
    + analysis["SDMVSTRA"].astype(str)
    + "__"
    + analysis["SDMVPSU"].astype(str)
)

sample_audit = (
    analysis.groupby(
        "NHANES_CYCLE",
        observed=True,
    )
    .agg(
        n=("SEQN", "size"),
        weighted_population_sum=(
            "WTSAF4YR",
            "sum",
        ),
        age_topcoded_n=(
            "age_topcoded",
            "sum",
        ),
        age_below_20_n=(
            "age_below_20",
            "sum",
        ),
        creatinine_min_mg_dL=(
            "creatinine_harmonized_mg_dL",
            "min",
        ),
        creatinine_median_mg_dL=(
            "creatinine_harmonized_mg_dL",
            "median",
        ),
        creatinine_max_mg_dL=(
            "creatinine_harmonized_mg_dL",
            "max",
        ),
    )
    .reset_index()
)

display(sample_audit.round(6))
print(f"Total analytic rows: {len(analysis):,}")


,NHANES_CYCLE,n,weighted_population_sum,age_topcoded_n,age_below_20_n,creatinine_min_mg_dL,creatinine_median_mg_dL,creatinine_max_mg_dL
0,2015_2016,2645,1.292913e+08,121,464,0.37056,0.817765,9.79041
1,2017_2018,2578,1.302142e+08,151,392,0.25000,0.820000,11.46000


Total analytic rows: 5,223


## 2. Define the sensitivity scenarios and stable formula

The bias values are added in native creatinine units (`mg/dL`) and then converted to the formula unit (`µmol/L`) using the governed factor `×88.4`.

The notebook calculates both existing diagnostic constant variants:

- `erratum`
- `supplement`

The sensitivity does not create a new formula coefficient set.


In [4]:
coefficients = CONFIG["formula"]["xb_coefficients"]
mortality_constants = CONFIG["formula"]["mortality_constants"]
formula_variants = CONFIG["formula"]["diagnostic_variants"]

SCENARIOS_MG_DL = {
    "observed_modern": 0.00,
    "nhanes3_bias_low": 0.11,
    "nhanes3_bias_mid": 0.17,
    "nhanes3_bias_high": 0.23,
}

CREATININE_CONVERSION = 88.4

expected_variants = {"erratum", "supplement"}

if not expected_variants.issubset(
    formula_variants
):
    raise RuntimeError(
        "The expected erratum and supplement variants "
        "are not both present in the configuration."
    )


def stable_phenoage_from_creatinine(
    frame: pd.DataFrame,
    *,
    creatinine_umol_column: str,
    variant_name: str,
) -> pd.Series:
    variant = formula_variants[variant_name]

    xb = (
        coefficients["intercept"]
        + coefficients["albumin"]
        * frame["albumin_harmonized_g_L"]
        + coefficients["creatinine"]
        * frame[creatinine_umol_column]
        + coefficients["glucose"]
        * frame["glucose_mmol_L"]
        + coefficients["log_crp"]
        * frame["log_crp_harmonized"]
        + coefficients["lymphocyte_percent"]
        * frame["lymphocyte_percent"]
        + coefficients["mcv"]
        * frame["mcv_fL"]
        + coefficients["rdw"]
        * frame["rdw_percent"]
        + coefficients["alp"]
        * frame["alp_harmonized_U_L"]
        + coefficients["wbc"]
        * frame["wbc_1000cells_uL"]
        + coefficients["chronological_age"]
        * frame["chronological_age_years"]
    )

    log_hazard = (
        np.log(
            mortality_constants["numerator"]
            / mortality_constants["denominator"]
        )
        + xb
    )

    return (
        variant["intercept"]
        + (
            np.log(-variant["multiplier"])
            + log_hazard
        )
        / variant["denominator"]
    )


## 3. Calculate participant-level scenario results


In [5]:
for scenario_name, shift_mg_dL in SCENARIOS_MG_DL.items():
    mg_column = (
        f"eg004_{scenario_name}_creatinine_mg_dL"
    )
    umol_column = (
        f"eg004_{scenario_name}_creatinine_umol_L"
    )

    analysis[mg_column] = (
        analysis["creatinine_harmonized_mg_dL"]
        + shift_mg_dL
    )
    analysis[umol_column] = (
        analysis[mg_column] * CREATININE_CONVERSION
    )

    if analysis[mg_column].le(0).any():
        raise RuntimeError(
            f"Scenario {scenario_name} produced "
            "non-positive creatinine values."
        )

    for variant_name in sorted(expected_variants):
        output_column = (
            f"eg004_{scenario_name}_"
            f"phenoage_{variant_name}_years"
        )

        analysis[output_column] = (
            stable_phenoage_from_creatinine(
                analysis,
                creatinine_umol_column=umol_column,
                variant_name=variant_name,
            )
        )

        reference_column = (
            f"harmonized_phenoage_{variant_name}_years"
        )
        delta_column = (
            f"eg004_{scenario_name}_"
            f"delta_{variant_name}_years"
        )

        analysis[delta_column] = (
            analysis[output_column]
            - analysis[reference_column]
        )

reference_checks = []

for variant_name in sorted(expected_variants):
    calculated = analysis[
        f"eg004_observed_modern_"
        f"phenoage_{variant_name}_years"
    ]
    existing = analysis[
        f"harmonized_phenoage_{variant_name}_years"
    ]

    maximum_absolute_difference = float(
        np.max(
            np.abs(
                calculated.to_numpy()
                - existing.to_numpy()
            )
        )
    )

    reference_checks.append(
        {
            "formula_variant": variant_name,
            "maximum_absolute_reference_difference": (
                maximum_absolute_difference
            ),
            "tolerance": 1e-10,
            "pass": maximum_absolute_difference < 1e-10,
        }
    )

reference_check = pd.DataFrame(reference_checks)

if not reference_check["pass"].all():
    raise RuntimeError(
        "The zero-shift reference does not reproduce the "
        "existing harmonized Phenotypic Age values."
    )

for variant_name in sorted(expected_variants):
    previous_delta = None

    for scenario_name in [
        "observed_modern",
        "nhanes3_bias_low",
        "nhanes3_bias_mid",
        "nhanes3_bias_high",
    ]:
        delta = analysis[
            f"eg004_{scenario_name}_"
            f"delta_{variant_name}_years"
        ]

        if not np.isfinite(
            delta.to_numpy(dtype=float)
        ).all():
            raise RuntimeError(
                f"Non-finite delta for {scenario_name}, "
                f"{variant_name}."
            )

        if previous_delta is not None:
            if (delta < previous_delta - 1e-12).any():
                raise RuntimeError(
                    "Creatinine sensitivity deltas are not "
                    f"monotonic for {variant_name}."
                )

        previous_delta = delta

display(reference_check)


,formula_variant,maximum_absolute_reference_difference,tolerance,pass
0,erratum,0.0,1.000000e-10,True
1,supplement,0.0,1.000000e-10,True


## 4. Survey-weighted summaries

Taylor-linearized means use pooled cycle-specific stratum and PSU identifiers. No lonely-PSU correction is applied.


In [6]:
def valid_weighted_frame(
    frame: pd.DataFrame,
    columns: list[str],
    weight: str = "WTSAF4YR",
) -> pd.DataFrame:
    valid = frame.loc[
        :,
        [*columns, weight],
    ].dropna().copy()

    return valid.loc[valid[weight] > 0]


def weighted_sd(
    frame: pd.DataFrame,
    value: str,
    weight: str = "WTSAF4YR",
) -> float:
    valid = valid_weighted_frame(
        frame,
        [value],
        weight,
    )

    mean = np.average(
        valid[value],
        weights=valid[weight],
    )
    variance = np.average(
        (valid[value] - mean) ** 2,
        weights=valid[weight],
    )

    return float(np.sqrt(variance))


def weighted_quantile(
    frame: pd.DataFrame,
    value: str,
    quantile: float,
    weight: str = "WTSAF4YR",
) -> float:
    valid = valid_weighted_frame(
        frame,
        [value],
        weight,
    ).sort_values(value)

    cumulative = valid[weight].cumsum()
    cutoff = quantile * valid[weight].sum()

    return float(
        valid.loc[
            cumulative.ge(cutoff),
            value,
        ].iloc[0]
    )


def survey_mean_taylor(
    frame: pd.DataFrame,
    value: str,
    *,
    weight: str = "WTSAF4YR",
    stratum: str = "pooled_stratum",
    psu: str = "pooled_psu",
) -> dict[str, float]:
    valid = frame.loc[
        :,
        [value, weight, stratum, psu],
    ].dropna().copy()

    valid = valid.loc[valid[weight] > 0]

    if valid.empty:
        raise RuntimeError(
            f"No valid observations for {value}."
        )

    estimate = float(
        np.average(
            valid[value],
            weights=valid[weight],
        )
    )
    total_weight = float(valid[weight].sum())

    valid["_linearized"] = (
        valid[weight]
        * (valid[value] - estimate)
        / total_weight
    )

    psu_totals = (
        valid.groupby(
            [stratum, psu],
            observed=True,
        )["_linearized"]
        .sum()
        .reset_index()
    )

    variance = 0.0

    for stratum_value, stratum_frame in psu_totals.groupby(
        stratum,
        observed=True,
    ):
        psu_values = stratum_frame[
            "_linearized"
        ].to_numpy(dtype=float)
        psu_count = len(psu_values)

        if psu_count < 2:
            raise RuntimeError(
                "A stratum has fewer than two observed PSUs: "
                f"{stratum_value}"
            )

        centered = (
            psu_values - psu_values.mean()
        )
        variance += (
            psu_count
            / (psu_count - 1)
            * np.square(centered).sum()
        )

    standard_error = float(np.sqrt(variance))

    return {
        "mean": estimate,
        "se": standard_error,
        "ci_low": estimate - 1.96 * standard_error,
        "ci_high": estimate + 1.96 * standard_error,
        "n": int(len(valid)),
        "weighted_population_sum": total_weight,
        "stratum_count": int(
            valid[stratum].nunique()
        ),
        "psu_count": int(
            valid[[stratum, psu]]
            .drop_duplicates()
            .shape[0]
        ),
    }


In [7]:
sample_masks = {
    "all_harmonized_complete_case": pd.Series(
        True,
        index=analysis.index,
    ),
    "no_topcode": ~analysis["age_topcoded"],
}

summary_records = []
quantile_records = []

for sample_name, sample_mask in sample_masks.items():
    sample_frame = analysis.loc[sample_mask]

    domains = {
        "pooled": sample_frame,
        **{
            cycle: cycle_frame
            for cycle, cycle_frame in sample_frame.groupby(
                "NHANES_CYCLE",
                observed=True,
            )
        },
    }

    for domain_name, domain_frame in domains.items():
        for variant_name in sorted(expected_variants):
            for (
                scenario_name,
                shift_mg_dL,
            ) in SCENARIOS_MG_DL.items():
                delta_column = (
                    f"eg004_{scenario_name}_"
                    f"delta_{variant_name}_years"
                )

                survey = survey_mean_taylor(
                    domain_frame,
                    delta_column,
                )

                summary_records.append(
                    {
                        "sample": sample_name,
                        "domain": domain_name,
                        "formula_variant": variant_name,
                        "scenario": scenario_name,
                        "creatinine_shift_mg_dL": (
                            shift_mg_dL
                        ),
                        "creatinine_shift_umol_L": (
                            shift_mg_dL
                            * CREATININE_CONVERSION
                        ),
                        "n": survey["n"],
                        "weighted_population_sum": (
                            survey[
                                "weighted_population_sum"
                            ]
                        ),
                        "weighted_mean_delta_years": (
                            survey["mean"]
                        ),
                        "taylor_se": survey["se"],
                        "ci_low_95": survey["ci_low"],
                        "ci_high_95": survey["ci_high"],
                        "weighted_sd_descriptive": (
                            weighted_sd(
                                domain_frame,
                                delta_column,
                            )
                        ),
                        "unweighted_mean_delta_years": float(
                            domain_frame[
                                delta_column
                            ].mean()
                        ),
                        "unweighted_min_delta_years": float(
                            domain_frame[
                                delta_column
                            ].min()
                        ),
                        "unweighted_max_delta_years": float(
                            domain_frame[
                                delta_column
                            ].max()
                        ),
                        "stratum_count": survey[
                            "stratum_count"
                        ],
                        "psu_count": survey["psu_count"],
                    }
                )

                quantile_records.append(
                    {
                        "sample": sample_name,
                        "domain": domain_name,
                        "formula_variant": variant_name,
                        "scenario": scenario_name,
                        "creatinine_shift_mg_dL": (
                            shift_mg_dL
                        ),
                        "weighted_q05_delta_years": (
                            weighted_quantile(
                                domain_frame,
                                delta_column,
                                0.05,
                            )
                        ),
                        "weighted_q25_delta_years": (
                            weighted_quantile(
                                domain_frame,
                                delta_column,
                                0.25,
                            )
                        ),
                        "weighted_median_delta_years": (
                            weighted_quantile(
                                domain_frame,
                                delta_column,
                                0.50,
                            )
                        ),
                        "weighted_q75_delta_years": (
                            weighted_quantile(
                                domain_frame,
                                delta_column,
                                0.75,
                            )
                        ),
                        "weighted_q95_delta_years": (
                            weighted_quantile(
                                domain_frame,
                                delta_column,
                                0.95,
                            )
                        ),
                    }
                )

scenario_summary = pd.DataFrame(
    summary_records
)
scenario_quantiles = pd.DataFrame(
    quantile_records
)

display(
    scenario_summary.loc[
        scenario_summary["domain"].eq("pooled")
        & scenario_summary["scenario"].ne(
            "observed_modern"
        )
    ].round(6)
)

display(
    scenario_quantiles.loc[
        scenario_quantiles["domain"].eq("pooled")
        & scenario_quantiles["scenario"].ne(
            "observed_modern"
        )
    ].round(6)
)


,sample,domain,formula_variant,scenario,creatinine_shift_mg_dL,creatinine_shift_umol_L,n,weighted_population_sum,weighted_mean_delta_years,taylor_se,ci_low_95,ci_high_95,weighted_sd_descriptive,unweighted_mean_delta_years,unweighted_min_delta_years,unweighted_max_delta_years,stratum_count,psu_count
1,all_harmonized_complete_case,pooled,erratum,nhanes3_bias_low,0.11,9.724,5223,2.595055e+08,1.007943,0.0,1.007943,1.007943,0.0,1.007943,1.007943,1.007943,30,60
2,all_harmonized_complete_case,pooled,erratum,nhanes3_bias_mid,0.17,15.028,5223,2.595055e+08,1.557730,0.0,1.557730,1.557730,0.0,1.557730,1.557730,1.557730,30,60
3,all_harmonized_complete_case,pooled,erratum,nhanes3_bias_high,0.23,20.332,5223,2.595055e+08,2.107518,0.0,2.107518,2.107518,0.0,2.107518,2.107518,2.107518,30,60
5,all_harmonized_complete_case,pooled,supplement,nhanes3_bias_low,0.11,9.724,5223,2.595055e+08,1.024544,0.0,1.024544,1.024544,0.0,1.024544,1.024544,1.024544,30,60
6,all_harmonized_complete_case,pooled,supplement,nhanes3_bias_mid,0.17,15.028,5223,2.595055e+08,1.583386,0.0,1.583386,1.583386,0.0,1.583386,1.583386,1.583386,30,60
7,all_harmonized_complete_case,pooled,supplement,nhanes3_bias_high,0.23,20.332,5223,2.595055e+08,2.142228,0.0,2.142228,2.142228,0.0,2.142228,2.142228,2.142228,30,60
25,no_topcode,pooled,erratum,nhanes3_bias_low,0.11,9.724,4951,2.504100e+08,1.007943,0.0,1.007943,1.007943,0.0,1.007943,1.007943,1.007943,30,60
26,no_topcode,pooled,erratum,nhanes3_bias_mid,0.17,15.028,4951,2.504100e+08,1.557730,0.0,1.557730,1.557730,0.0,1.557730,1.557730,1.557730,30,60
27,no_topcode,pooled,erratum,nhanes3_bias_high,0.23,20.332,4951,2.504100e+08,2.107518,0.0,2.107518,2.107518,0.0,2.107518,2.107518,2.107518,30,60
29,no_topcode,pooled,supplement,nhanes3_bias_low,0.11,9.724,4951,2.504100e+08,1.024544,0.0,1.024544,1.024544,0.0,1.024544,1.024544,1.024544,30,60


,sample,domain,formula_variant,scenario,creatinine_shift_mg_dL,weighted_q05_delta_years,weighted_q25_delta_years,weighted_median_delta_years,weighted_q75_delta_years,weighted_q95_delta_years
1,all_harmonized_complete_case,pooled,erratum,nhanes3_bias_low,0.11,1.007943,1.007943,1.007943,1.007943,1.007943
2,all_harmonized_complete_case,pooled,erratum,nhanes3_bias_mid,0.17,1.557730,1.557730,1.557730,1.557730,1.557730
3,all_harmonized_complete_case,pooled,erratum,nhanes3_bias_high,0.23,2.107518,2.107518,2.107518,2.107518,2.107518
5,all_harmonized_complete_case,pooled,supplement,nhanes3_bias_low,0.11,1.024544,1.024544,1.024544,1.024544,1.024544
6,all_harmonized_complete_case,pooled,supplement,nhanes3_bias_mid,0.17,1.583386,1.583386,1.583386,1.583386,1.583386
7,all_harmonized_complete_case,pooled,supplement,nhanes3_bias_high,0.23,2.142228,2.142228,2.142228,2.142228,2.142228
25,no_topcode,pooled,erratum,nhanes3_bias_low,0.11,1.007943,1.007943,1.007943,1.007943,1.007943
26,no_topcode,pooled,erratum,nhanes3_bias_mid,0.17,1.557730,1.557730,1.557730,1.557730,1.557730
27,no_topcode,pooled,erratum,nhanes3_bias_high,0.23,2.107518,2.107518,2.107518,2.107518,2.107518
29,no_topcode,pooled,supplement,nhanes3_bias_low,0.11,1.024544,1.024544,1.024544,1.024544,1.024544


## 5. Decision-support interpretation

This section quantifies materiality but does not decide whether the compensating shift is scientifically correct.


In [8]:
decision_table = scenario_summary.loc[
    scenario_summary["sample"].eq(
        "all_harmonized_complete_case"
    )
    & scenario_summary["domain"].eq("pooled")
    & scenario_summary["scenario"].ne(
        "observed_modern"
    ),
    [
        "formula_variant",
        "scenario",
        "creatinine_shift_mg_dL",
        "weighted_mean_delta_years",
        "taylor_se",
        "ci_low_95",
        "ci_high_95",
        "weighted_sd_descriptive",
        "unweighted_min_delta_years",
        "unweighted_max_delta_years",
    ],
].copy()

decision_table["materiality_category"] = np.select(
    [
        decision_table[
            "weighted_mean_delta_years"
        ].abs().ge(1.0),
        decision_table[
            "weighted_mean_delta_years"
        ].abs().ge(0.5),
    ],
    [
        "material_at_least_1_year",
        "material_0.5_to_1_year",
    ],
    default="below_0.5_year",
)

if not decision_table[
    "weighted_mean_delta_years"
].gt(0).all():
    raise RuntimeError(
        "A positive NHANES-III-scale shift did not increase "
        "Phenotypic Age as expected from the positive "
        "creatinine coefficient."
    )

display(decision_table.round(6))


,formula_variant,scenario,creatinine_shift_mg_dL,weighted_mean_delta_years,taylor_se,ci_low_95,ci_high_95,weighted_sd_descriptive,unweighted_min_delta_years,unweighted_max_delta_years,materiality_category
1,erratum,nhanes3_bias_low,0.11,1.007943,0.0,1.007943,1.007943,0.0,1.007943,1.007943,material_at_least_1_year
2,erratum,nhanes3_bias_mid,0.17,1.557730,0.0,1.557730,1.557730,0.0,1.557730,1.557730,material_at_least_1_year
3,erratum,nhanes3_bias_high,0.23,2.107518,0.0,2.107518,2.107518,0.0,2.107518,2.107518,material_at_least_1_year
5,supplement,nhanes3_bias_low,0.11,1.024544,0.0,1.024544,1.024544,0.0,1.024544,1.024544,material_at_least_1_year
6,supplement,nhanes3_bias_mid,0.17,1.583386,0.0,1.583386,1.583386,0.0,1.583386,1.583386,material_at_least_1_year
7,supplement,nhanes3_bias_high,0.23,2.142228,0.0,2.142228,2.142228,0.0,2.142228,2.142228,material_at_least_1_year


## 6. Write outputs and the EG-004 report draft


In [9]:
def markdown_table(
    frame: pd.DataFrame,
) -> str:
    columns = [str(column) for column in frame.columns]
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(
        ["---"] * len(columns)
    ) + " |"

    rows = []

    for _, row in frame.iterrows():
        values = []

        for value in row:
            if pd.isna(value):
                text = ""
            elif isinstance(
                value,
                (float, np.floating),
            ):
                text = f"{float(value):.6g}"
            else:
                text = str(value)

            values.append(
                text.replace("|", "\\|")
            )

        rows.append(
            "| " + " | ".join(values) + " |"
        )

    return "\n".join(
        [header, separator, *rows]
    )


PARTICIPANT_PATH = (
    PROCESSED_ROOT
    / "06_eg004_creatinine_sensitivity_diagnostic.parquet"
)
SUMMARY_PATH = (
    TABLES_ROOT
    / "06_eg004_creatinine_sensitivity_summary.csv"
)
QUANTILE_PATH = (
    TABLES_ROOT
    / "06_eg004_creatinine_sensitivity_quantiles.csv"
)
SAMPLE_AUDIT_PATH = (
    TABLES_ROOT
    / "06_eg004_sample_audit.csv"
)
REFERENCE_CHECK_PATH = (
    TABLES_ROOT
    / "06_eg004_reference_equivalence_check.csv"
)
REPORT_PATH = (
    DOCS_METHODOLOGY_ROOT
    / "EG004_Creatinine_Sensitivity_Report_Draft.md"
)
METADATA_PATH = (
    LOGS_ROOT
    / "06_eg004_creatinine_sensitivity_metadata.json"
)

participant_columns = [
    "SEQN",
    "NHANES_CYCLE",
    "chronological_age_years",
    "age_topcoded",
    "age_below_20",
    "WTSAF4YR",
    "SDMVSTRA",
    "SDMVPSU",
    "creatinine_harmonized_mg_dL",
    "creatinine_harmonized_umol_L",
    "harmonized_phenoage_erratum_years",
    "harmonized_phenoage_supplement_years",
]

for scenario_name in SCENARIOS_MG_DL:
    participant_columns.extend(
        [
            (
                f"eg004_{scenario_name}_"
                "creatinine_mg_dL"
            ),
            (
                f"eg004_{scenario_name}_"
                "creatinine_umol_L"
            ),
        ]
    )

    for variant_name in sorted(expected_variants):
        participant_columns.extend(
            [
                (
                    f"eg004_{scenario_name}_"
                    f"phenoage_{variant_name}_years"
                ),
                (
                    f"eg004_{scenario_name}_"
                    f"delta_{variant_name}_years"
                ),
            ]
        )

participant_output = analysis.loc[
    :,
    participant_columns,
].copy()
participant_output["diagnostic_only"] = True
participant_output["final_scientific_result"] = False

participant_output.to_parquet(
    PARTICIPANT_PATH,
    index=False,
)
scenario_summary.to_csv(
    SUMMARY_PATH,
    index=False,
)
scenario_quantiles.to_csv(
    QUANTILE_PATH,
    index=False,
)
sample_audit.to_csv(
    SAMPLE_AUDIT_PATH,
    index=False,
)
reference_check.to_csv(
    REFERENCE_CHECK_PATH,
    index=False,
)

report_table = decision_table[
    [
        "formula_variant",
        "scenario",
        "creatinine_shift_mg_dL",
        "weighted_mean_delta_years",
        "taylor_se",
        "ci_low_95",
        "ci_high_95",
        "materiality_category",
    ]
].copy()

report = f"""# EG-004 Creatinine Training-Scale Sensitivity Report — Draft

## Document Control

| Field | Value |
| --- | --- |
| Project | AgeLens |
| Evidence Gap | EG-004 |
| Status | Draft — diagnostic decision support |
| Generated At (UTC) | {datetime.now(timezone.utc).isoformat()} |
| Mortality Data Used | No |
| Governance Documents Modified | No |

## 1. Question

D-001 was trained using NHANES III creatinine measurements reported approximately 0.11–0.23 mg/dL higher than a standardized enzymatic reference. This analysis quantifies how much AgeLens Phenotypic Age changes when modern harmonized creatinine is shifted upward across that documented range.

The scenarios do not establish that a compensating adjustment is scientifically correct. They quantify the consequence of adopting one.

## 2. Sample

{markdown_table(sample_audit)}

All participants were positive-weight harmonized complete cases. No imputation was performed.

## 3. Scenarios

- Observed modern scale: +0.00 mg/dL.
- Lower NHANES III bias approximation: +0.11 mg/dL.
- Midpoint approximation: +0.17 mg/dL.
- Upper approximation: +0.23 mg/dL.

The shift was applied before conversion to µmol/L.

## 4. Pooled Survey-Weighted Effect

{markdown_table(report_table)}

## 5. Interpretation

A positive shift increases Phenotypic Age because the D-001 creatinine coefficient is positive. The analysis should be interpreted as a bounded sensitivity range, not as validation of a correction equation.

BioAge agreement cannot adjudicate EG-004: BioAge and AgeLens both receive the same modern creatinine input, so software parity favors the unshifted input by construction and does not reveal which laboratory scale is scientifically appropriate for a model trained on NHANES III.

## 6. Decision Options

1. **Unadjusted replication:** retain modern harmonized creatinine as observed and document the sensitivity range as an accepted limitation.
2. **Compensating adjustment:** adopt a specifically justified upward shift or source-derived transformation to approximate the NHANES III training scale.
3. **Postpone final release:** seek stronger direct evidence or recalibrate/rederive the model before choosing a primary scale.

No option is approved by this report.

## 7. Release Status

EG-004 remains open and final scientific results remain disabled. An explicit governance Decision is required before mortality/outcome analyses are treated as final.
"""

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

metadata = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "notebook": (
        "06_eg004_creatinine_sensitivity.ipynb"
    ),
    "input": str(
        INPUT_PATH.relative_to(PROJECT_ROOT)
    ),
    "evidence_gap": "EG-004",
    "scenario_shifts_mg_dL": SCENARIOS_MG_DL,
    "creatinine_conversion_to_umol_L": (
        CREATININE_CONVERSION
    ),
    "formula_variants": sorted(
        expected_variants
    ),
    "survey_weight": "WTSAF4YR",
    "survey_stratum": "pooled_stratum",
    "survey_psu": "pooled_psu",
    "lonely_psu_correction_applied": False,
    "mortality_data_used": False,
    "governance_changes_applied": False,
    "outputs_diagnostic_only": True,
    "final_scientific_results_allowed": False,
    "open_core_evidence_gaps": sorted(
        open_gaps
    ),
    "outputs": [
        str(
            path.relative_to(PROJECT_ROOT)
        )
        for path in [
            PARTICIPANT_PATH,
            SUMMARY_PATH,
            QUANTILE_PATH,
            SAMPLE_AUDIT_PATH,
            REFERENCE_CHECK_PATH,
            REPORT_PATH,
        ]
    ],
}

METADATA_PATH.write_text(
    json.dumps(
        metadata,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

required_outputs = [
    PARTICIPANT_PATH,
    SUMMARY_PATH,
    QUANTILE_PATH,
    SAMPLE_AUDIT_PATH,
    REFERENCE_CHECK_PATH,
    REPORT_PATH,
    METADATA_PATH,
]

missing_outputs = [
    path
    for path in required_outputs
    if not path.exists()
]

if missing_outputs:
    raise RuntimeError(
        f"Expected outputs were not created: {missing_outputs}"
    )

print("Files written:")
for path in required_outputs:
    print(f"  - {path.relative_to(PROJECT_ROOT)}")

print("✅ EG-004 sensitivity outputs verified.")
print("No governance decision was applied.")
print("Mortality data were not used.")
print("Final scientific results remain disabled.")


Files written:
  - data\processed\06_eg004_creatinine_sensitivity_diagnostic.parquet
  - results\tables\06_eg004_creatinine_sensitivity_summary.csv
  - results\tables\06_eg004_creatinine_sensitivity_quantiles.csv
  - results\tables\06_eg004_sample_audit.csv
  - results\tables\06_eg004_reference_equivalence_check.csv
  - docs\methodology\EG004_Creatinine_Sensitivity_Report_Draft.md
  - logs\06_eg004_creatinine_sensitivity_metadata.json
✅ EG-004 sensitivity outputs verified.
No governance decision was applied.
Mortality data were not used.
Final scientific results remain disabled.
